In [1]:
%load_ext autoreload
%autoreload 2

# Phase 1

In [ ]:
import os
import shutil
import sys
import gymnasium as gym
import torch

# Ensure repository root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
  sys.path.insert(0, PROJECT_ROOT)

from src.rl_transformer.env_adapter import MatchEnv
from src.rl_transformer.pool import PoolOpponentController
from src.rl_transformer.ppo import train_mappo
from src.rl_transformer.transformer_model import TransformerActorCritic

# ── Hardware Performance Flags ──
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── 3v3 Regulation Dimensions & Physics ──
TEAM_SIZE = 3
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0
ROUND_STEPS = 4500
ACTION_REPEAT = 6
NUM_ENVS = 24

# ── Directories & Seed Checkpoints ──
SAVE_DIR_S2_P3 = "models/stage3/phase1"
SAVE_DIR_S3_P1 = "models/stage3/phase1"
POOL_DIR_S3_P1 = os.path.join(SAVE_DIR_S3_P1, "pool")

os.makedirs(SAVE_DIR_S3_P1, exist_ok=True)
os.makedirs(POOL_DIR_S3_P1, exist_ok=True)

seed_file = os.path.join(SAVE_DIR_S2_P3, "best_model.pt")
if not os.path.exists(seed_file):
  seed_file = os.path.join(SAVE_DIR_S2_P3, "final_model.pt")

if not os.path.exists(seed_file):
  raise FileNotFoundError(f"Missing checkpoint to seed Stage 3: {seed_file}")

shutil.copy(seed_file, os.path.join(POOL_DIR_S3_P1, "champion.pt"))
shutil.copy(seed_file, os.path.join(POOL_DIR_S3_P1, "history_0.pt"))
print(f"🔥 Stage 3 Phase 1 seeded from: {seed_file}")


# ── Environment Factory ──
def make_s3_p1_env(env_rank: int):
  def _thunk():
    torch.set_num_threads(1)

    # 1. Group 1: 50% Heuristic with Random Reset (12 envs: 0-11)
    if env_rank < 12:
      if env_rank < 10:
        opp_team_size = 3
      elif env_rank == 10:
        opp_team_size = 4
      else:
        opp_team_size = 5
      p_random, p_heuristic = 0.0, 1.0
      random_reset_modes = ["heuristic"]

    # 2. Group 2: ~30% Heuristic with NO Random Reset (7 envs: 12-18)
    elif env_rank < 19:
      if env_rank < 17:
        opp_team_size = 3
      elif env_rank == 17:
        opp_team_size = 4
      else:
        opp_team_size = 5
      p_random, p_heuristic = 0.0, 1.0
      random_reset_modes = []

    # 3. Group 3: 10% Random (Random Reset) + 10% Self-Play (No Random Reset) (5 envs: 19-23)
    else:
      opp_team_size = 3
      p_random = 1.0     
      p_heuristic = 0.0   
      random_reset_modes = ["random"]  # Random triggers chaos reset; Self-play gets standard kickoff

    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S3_P1,
        team="blue",
        device="cpu",
        p_random=p_random,
        p_heuristic=p_heuristic,
        frame_stack=3,
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=opp_team_size,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
        opponent_stats=[
            (3200.0, 1200.0),  # Standard tier
        ],
        frame_stack=3,
        random_reset_opponents=random_reset_modes,
    )
    env.reset(seed=7000 + env_rank)
    return env

  return _thunk


envs_s3_p1 = gym.vector.AsyncVectorEnv(
    [make_s3_p1_env(i) for i in range(NUM_ENVS)],
    context="fork",
    shared_memory=False,
)

# ── Model Initialization & Warmstart ──
model_s3_p1 = TransformerActorCritic().to(device)
ckpt = torch.load(seed_file, map_location=device, weights_only=False)
state_dict = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)
model_s3_p1.load_state_dict(state_dict, strict=True)
print("✅ Weights successfully transferred into Stage 3 model.")

# ── Training Loop ──
train_mappo(
    envs=envs_s3_p1,
    model=model_s3_p1,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,  # Official evaluation benchmark remains regulation 3v3
    total_timesteps=20_000_000,
    num_envs=NUM_ENVS,
    num_steps=384,
    update_epochs=3,
    minibatch_size=2048,
    lr_init=3e-5,
    lr_final=3e-6,
    ent_coef_init=0.010,
    ent_coef_final=0.004,
    gamma=0.997,
    gae_lambda=0.97,
    active_tiers=["heuristic"],
    target_tier="heuristic",
    eval_episodes=50,
    eval_freq=250_000,
    save_dir=SAVE_DIR_S3_P1,
    pool_dir=POOL_DIR_S3_P1,
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s3_p1.close()

In [ ]:
import os
import torch
import sys
# Ensure repository root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
  sys.path.insert(0, PROJECT_ROOT)

from src.rl_transformer.visualization import evaluate_and_generate_html



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

html_path = evaluate_and_generate_html(
    red_agent="models/stage3/phase1/final_model.pt",
    blue_agent="heuristic",
    red_team_size=3,
    blue_team_size=5,
    device=device,
    filename="stage3_phase1_heuristic.html",
    num_episodes=10,
    max_steps=4500,
    action_repeat=6,
    pitch_width=1200.0,
    pitch_height=800.0,
    goal_height=220.0,
)

# Phase 2

In [ ]:
import os
import shutil
import sys
import gymnasium as gym
import torch

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
  sys.path.insert(0, PROJECT_ROOT)

from src.rl_transformer.env_adapter import MatchEnv
from src.rl_transformer.pool import PoolOpponentController
from src.rl_transformer.ppo import train_mappo
from src.rl_transformer.transformer_model import TransformerActorCritic

torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── 3v3 Regulation Dimensions & Physics ──
TEAM_SIZE = 3
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0
ROUND_STEPS = 4500
ACTION_REPEAT = 6
NUM_ENVS = 24

# ── Directories & Fresh Seed ──
SAVE_DIR_S3_P1 = "models/stage3/phase1"
SAVE_DIR_S3_P2 = "models/stage3/phase2"
POOL_DIR_S3_P2 = os.path.join(SAVE_DIR_S3_P2, "pool")

os.makedirs(SAVE_DIR_S3_P2, exist_ok=True)
os.makedirs(POOL_DIR_S3_P2, exist_ok=True)

seed_file = os.path.join(SAVE_DIR_S3_P1, "final_model.pt")
if not os.path.exists(seed_file):
  seed_file = os.path.join(SAVE_DIR_S3_P1, "best_model.pt")

shutil.copy(seed_file, os.path.join(POOL_DIR_S3_P2, "champion.pt"))
shutil.copy(seed_file, os.path.join(POOL_DIR_S3_P2, "history_0.pt"))
print(f"🔥 Stage 3 Phase 2 seeded cleanly from: {seed_file}")


# ── Environment Factory (50% Heuristic Anchor, 45% Self-Play, 5% Random) ──
def make_s3_p2_env(env_rank: int):
  def _thunk():
    torch.set_num_threads(1)

    # 12 Envs (50%): Heuristic Anchor (Structured kickoff)
    if env_rank < 12:
      opp_team_size = 3
      p_random = 0.0
      p_heuristic = 1.0

    # 12 Envs (50%): 45% Self-Play Pool + 5% Random
    else:
      opp_team_size = 3
      p_random = 0.10   # 0.10 * (12/24) = 5.0% Total Random
      p_heuristic = 0.0  # Remaining 90% of 12 envs = 45.0% Self-Play

    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S3_P2,
        team="blue",
        device="cpu",
        p_random=p_random,
        p_heuristic=p_heuristic,
        frame_stack=3,
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=opp_team_size,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
        opponent_stats=[(3200.0, 1200.0)],
        frame_stack=3,
        random_reset_opponents=[],  # Standard kickoffs maintain consistent formations
    )
    env.reset(seed=8000 + env_rank)
    return env

  return _thunk


envs_s3_p2 = gym.vector.AsyncVectorEnv(
    [make_s3_p2_env(i) for i in range(NUM_ENVS)],
    context="fork",
    shared_memory=False,
)

# ── Model Initialization ──
model_s3_p2 = TransformerActorCritic().to(device)
ckpt = torch.load(seed_file, map_location=device, weights_only=False)
state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
model_s3_p2.load_state_dict(state_dict, strict=True)
print("✅ Weights successfully loaded into Stage 3 Phase 2 model.")

# ── Training Loop ──
train_mappo(
    envs=envs_s3_p2,
    model=model_s3_p2,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,
    total_timesteps=50_000_000,
    num_envs=NUM_ENVS,
    num_steps=384,
    update_epochs=3,
    minibatch_size=2048,
    lr_init=8e-6,           # Stable fine-tuning LR (avoids policy collapse)
    lr_final=2e-6,
    ent_coef_init=0.003,    # Low entropy preserves disciplined defensive spacing
    ent_coef_final=0.0008,
    gamma=0.996,
    gae_lambda=0.96,
    active_tiers=["heuristic", "champion"],
    target_tier="champion",
    filter_thresholds={"heuristic": 0.75},
    tier_ratios={"heuristic": 0.40, "champion": 0.60},
    eval_episodes=80,
    eval_freq=500_000,
    save_dir=SAVE_DIR_S3_P2,
    pool_dir=POOL_DIR_S3_P2,
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s3_p2.close()

Using device: cuda
🔥 Stage 3 Phase 2 seeded cleanly from: models/stage3/phase1/final_model.pt
✅ Weights successfully loaded into Stage 3 Phase 2 model.
🚀 Entity-Transformer MAPPO Initialized | Format: 3v3 | Envs: 24 | Batch: 27648 | Device: cuda

📊 [EVALUATION @ Step 525,312 | Rollout SPS: 3624 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  87.5% | Reward: +3.288 | Goals: 73 Scored, 1 Conceded (+72 Net)
   ⚔️  vs Champion  [TARGET] | WR:  22.9% | Reward: -0.427 | Goals: 13 Scored, 18 Conceded (-5 Net)
   ❌ Retaining current baseline. Did not pass criteria for champion: [WR: 22.9%, Net: -5, Reward: -0.427] (Eval took 34.4s)

📊 [EVALUATION @ Step 1,022,976 | Rollout SPS: 3632 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  81.2% | Reward: +2.816 | Goals: 63 Scored, 2 Conceded (+61 Net)
   ⚔️  vs Champion  [TARGET] | WR:  22.9% | Reward: -0.158 | Goals: 18 Scored, 14 Conceded (+4 Net)
   ❌ Retaining current baseline. Did not pass criteria f

In [ ]:
import os
import torch
import sys
# Ensure repository root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
  sys.path.insert(0, PROJECT_ROOT)

from src.rl_transformer.visualization import evaluate_and_generate_html



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

html_path = evaluate_and_generate_html(
    red_agent="models/stage3/phase2/best_model.pt",
    blue_agent="heuristic",
    red_team_size=3,
    blue_team_size=5,
    device=device,
    filename="stage3_phase2_heuristic.html",
    num_episodes=10,
    max_steps=4500,
    action_repeat=6,
    pitch_width=1200.0,
    pitch_height=800.0,
    goal_height=220.0,
)